# Tujuan

Notebook ini bertanggung jawab untuk:

1. Preparasi & Verifikasi File Sumber
    - Memvalidasi source file (existence, tipe, ukuran).
    - Mengambil fingerprint source menggunakan SHA-256.
    - Memvalidasi struktur CSV mentah (encoding, delimiter, duplicate header).
    - Membaca raw dataset.

2. Standarisasi & Validasi Struktur Data
    - Menstandarkan nama kolom ke lowercase.
    - Memvalidasi schema dan data type terhadap satu *schema contract* tunggal.
    - Memvalidasi primary key (`passengerid`).
    - Memvalidasi duplicate record (full-row **dan** business-key).

3. Validasi Kualitas & Domain Data
    - Membuat profiling missing values & memvalidasi threshold missingness.
    - Memvalidasi kualitas string (empty / whitespace) dan format nama.
    - Memvalidasi domain numerik dan domain kategorikal.
    - Memvalidasi hubungan train/test (ID overlap) dan kewajaran jumlah baris.

4. Penyimpanan & Verifikasi Artefak (Staging)
    - Menyimpan dataset ke staged Parquet.
    - Memvalidasi ulang staged artifact (read-back + schema fingerprint match).

5. Dokumentasi, Metadata & Provenance
    - Membuat validation report (JSON + Markdown) yang **selalu tertulis**, baik proses lolos maupun gagal.
    - Membuat ingestion metadata lengkap.
    - Mencatat provenance (git) dan environment.

6. Pembersihan Sistem
    - Membersihkan resource setelah proses selesai.

# Boundary

Notebook ini **tidak melakukan data cleaning substantif**.

Tidak dilakukan:

- imputasi missing value
- penghapusan duplicate
- penghapusan outlier
- encoding
- feature engineering
- perubahan nilai data
- koreksi data source

Normalisasi nama kolom ke lowercase hanya merupakan standardisasi schema teknis. Jika ditemukan masalah pada raw dataset, masalah tersebut **dideteksi dan dilaporkan**, bukan diperbaiki di notebook ini.

# Preparation & Verification

## Import & Configuration

In [1]:
from pathlib import Path
from datetime import datetime, timedelta, timezone
import time
import sys
import json

import polars as pl

# Polars Config
pl.Config.set_tbl_rows(-1)
pl.Config.set_tbl_cols(-1)

# Path Project
PROJECT_ROOT = Path.cwd().parent if Path.cwd().name == "notebooks" else Path.cwd()

if str(PROJECT_ROOT) not in sys.path:
    sys.path.append(str(PROJECT_ROOT))

print(PROJECT_ROOT)

d:\GIT DATA\titanic_survival_analysis


## Path Configuration

In [2]:
# Raw location
RAW_DIR = PROJECT_ROOT / "data" / "raw"
TRAIN_RAW = RAW_DIR / "train.csv"
TEST_RAW = RAW_DIR / "test.csv"

# Staging location
STAGED_DIR = PROJECT_ROOT / "data" / "stage"

# Parquet
TRAIN_PARQUET = STAGED_DIR / "train.parquet"
TEST_PARQUET = STAGED_DIR / "test.parquet"

# Dir
STAGED_DIR.mkdir(parents = True, exist_ok = True)


print(f"TRAIN_RAW  : {TRAIN_RAW}")
print(f"TEST_RAW   : {TEST_RAW} \n")

print(f"STAGE_DIR : {STAGED_DIR}")

TRAIN_RAW  : d:\GIT DATA\titanic_survival_analysis\data\raw\train.csv
TEST_RAW   : d:\GIT DATA\titanic_survival_analysis\data\raw\test.csv 

STAGE_DIR : d:\GIT DATA\titanic_survival_analysis\data\stage


## Contract

In [3]:
DATASET_NAME = "titanic"
PIPELINE_STAGE = "ingestion"
PIPELINE_VERSION = "1.0.0"
SCHEMA_VERSION = "1.0.0"
DQ_RULES_VERSION = "1.0.0"
SOURCE_FORMAT = "csv"
TARGET_FORMAT = "parquet"

print(f"Dataset name        : {DATASET_NAME}")
print(f"Pipeline stage      : {PIPELINE_STAGE}")
print(f"Pipeline version    : {PIPELINE_VERSION}")
print(f"Schema version      : {SCHEMA_VERSION}")
print(f"DQ rules version    : {DQ_RULES_VERSION} \n")

print(f"Source format       : {SOURCE_FORMAT}")
print(f"Target format       : {TARGET_FORMAT}")


Dataset name        : titanic
Pipeline stage      : ingestion
Pipeline version    : 1.0.0
Schema version      : 1.0.0
DQ rules version    : 1.0.0 

Source format       : csv
Target format       : parquet


## Run Identification

In [4]:
# Waktu awal
INGESTION_START = time.perf_counter()

# Timezone
WIB = timezone(timedelta(hours=7))
RUN_TIMESTAMP = datetime.now(WIB)

# Format custom
STARTED_FORMATTED = RUN_TIMESTAMP.strftime("%Y-%m-%d %H:%M:%S WIB")

RUN_ID = f"{DATASET_NAME}-{PIPELINE_STAGE}-{STARTED_FORMATTED}"


print(f"Run ID  : {RUN_ID}")
print(f"Started : {STARTED_FORMATTED}")

Run ID  : titanic-ingestion-2026-09-23 15:03:04 WIB
Started : 2026-09-23 15:03:04 WIB


# Blueprint - Sudah cek manual (nama kolom, data type, dsb)

| Variable      | Definition                | Key                   | Role          | (Required, Null)  | Reason                                      |
| ------------- | ------------------------- | --------------------- | ------------- | ----------------- | ------------------------------------------- |
| PassengerId   | Table Identifier          |                       | Primary Key   | (True, False)     | Unique ID data                              |
| Survived      | Survival                  | `0` = No, `1` = Yes   | Target        | (True, False)     | Target result                               |
| Pclass        | Ticket Class              | `1` = 1st (Upper),    | Category      | (True, False)     | Customer category                           |
|               |                           | `2` = 2nd (Middle),   |               |                   |                                             |
|               |                           | `3` = 3rd (Lower)     |               |                   |                                             |
| Name          | Passenger Name            |                       | Text          | (True, False)     | Personal customers data                     |
| Sex           | Sex                       |                       | Category      | (True, False)     | Personal customers data                     |
| Age           | Age in years              |                       | Numeric       | (False, True)     | There was null data                         |
| SibSp         | siblings / spouses aboard |                       | Numeric       | (True, False)     | Feature engineering (for travel group size) |
| Parch         | parents / children aboard |                       | Numeric       | (True, False)     | Feature engineering (for travel group size) |
| Ticket        | Ticket number             |                       | Text          | (True, False)     | Customer must have ticket                   |
| Fare          | Passenger Fee             |                       | Numeric       | (False, True)     | There was null data                         |
| Cabin         | Cabin number              |                       | Text          | (False, True)     | There was null data                         |
| Embarked      | Port of Embarkation       | `C` = Cherbourg,      | Category      | (False, True)     | There was null data                         |
|               |                           | `Q` = Queenstown,     |               |                   |                                             |
|               |                           | `S` = Southampton     |               |                   |                                             |

## Required Schema

In [5]:
REQUIRED_SCHEMA_CONTRACT = {
    "passengerid": {
        "dtype": "Int64",
        "semantic_type": "primary_key",
    },
    "survived": {
        "dtype": "Int64",
        "semantic_type": "binary_target",
        "allowed_values": [0, 1],
    },
    "pclass": {
        "dtype": "Int64",
        "semantic_type": "categorical",
        "allowed_values": [1, 2, 3],
    },
    "name": {
        "dtype": "String",
        "semantic_type": "text",
    },
    "sex": {
        "dtype": "String",
        "semantic_type": "categorical",
        "allowed_values": ["male", "female"],
    },
    "sibsp": {
        "dtype": "Int64",
        "semantic_type": "count",
        "min": 0,
    },
    "parch": {
        "dtype": "Int64",
        "semantic_type": "count",
        "min": 0,
    },
    "ticket": {
        "dtype": "String",
        "semantic_type": "identifier",
    },
}

### Other Schema

In [6]:
OTHER_SCHEMA_CONTRACT = {
    "age": {
        "dtype": "Float64",
        "semantic_type": "numeric",
        "min": 0,
        "max": 100,
        "max_null_ratio": 0.3,
    },
    "fare": {
        "dtype": "Float64",
        "semantic_type": "numeric",
        "min": 0,
        "max_null_ratio": 0.05,
    },
    "cabin": {
        "dtype": "String",
        "semantic_type": "categorical_text",
        "max_null_ratio": 0.9,
    },
    "embarked": {
        "dtype": "String",
        "semantic_type": "categorical",
        "allowed_values": ["C", "Q", "S"],
        "max_null_ratio": 0.05,
    },
}

## Schema Contract

In [7]:
# 
SCHEMA_CONTRACT = {}

# filter for required
for col, spec in REQUIRED_SCHEMA_CONTRACT.items():
    rule = spec.copy()
    rule["nullable"] = False
    rule["required"] = True
    SCHEMA_CONTRACT[col] = rule

# filter for other
for col, spec in OTHER_SCHEMA_CONTRACT.items():
    rule = spec.copy()
    rule["nullable"] = True
    rule["required"] = False
    SCHEMA_CONTRACT[col] = rule

In [8]:
print(f"✓ REQUIRED_SCHEMA_CONTRACT : {len(REQUIRED_SCHEMA_CONTRACT)} kolom")
print(f"✓ OTHER_SCHEMA_CONTRACT    : {len(OTHER_SCHEMA_CONTRACT)} kolom \n")

print(f"✓ Total SCHEMA_CONTRACT    : {len(SCHEMA_CONTRACT)} kolom \n")
print(f"--- ISI SCHEMA_CONTRACT ---")
print(json.dumps(SCHEMA_CONTRACT, indent = 4))

✓ REQUIRED_SCHEMA_CONTRACT : 8 kolom
✓ OTHER_SCHEMA_CONTRACT    : 4 kolom 

✓ Total SCHEMA_CONTRACT    : 12 kolom 

--- ISI SCHEMA_CONTRACT ---
{
    "passengerid": {
        "dtype": "Int64",
        "semantic_type": "primary_key",
        "nullable": false,
        "required": true
    },
    "survived": {
        "dtype": "Int64",
        "semantic_type": "binary_target",
        "allowed_values": [
            0,
            1
        ],
        "nullable": false,
        "required": true
    },
    "pclass": {
        "dtype": "Int64",
        "semantic_type": "categorical",
        "allowed_values": [
            1,
            2,
            3
        ],
        "nullable": false,
        "required": true
    },
    "name": {
        "dtype": "String",
        "semantic_type": "text",
        "nullable": false,
        "required": true
    },
    "sex": {
        "dtype": "String",
        "semantic_type": "categorical",
        "allowed_values": [
            "male",
          

#### Filter Schema Contract

In [9]:
# List Shcema Order
COLUMN_ORDER = list(SCHEMA_CONTRACT.keys())

print(f"Urutan di Schema Contract: {COLUMN_ORDER}")

Urutan di Schema Contract: ['passengerid', 'survived', 'pclass', 'name', 'sex', 'sibsp', 'parch', 'ticket', 'age', 'fare', 'cabin', 'embarked']


In [10]:
# Default for most data
DEFAULT_TYPE = {
    "Int64": pl.Int64,
    "Float64": pl.Float64,
    "String": pl.String,
}

# check data type
EXPECTED_DTYPES = {
    column: DEFAULT_TYPE[spec["dtype"]]
    for column, spec in SCHEMA_CONTRACT.items()
}

# check string
STRING_COLUMNS = [
    column for column, spec in SCHEMA_CONTRACT.items() if spec["dtype"] == "String"
]

# check numeric
NUMERIC_BOUNDS = {
    column: {"min": spec.get("min"), "max": spec.get("max")}
    for column, spec in SCHEMA_CONTRACT.items()
    if spec["dtype"] in ("Int64", "Float64") and ("min" in spec or "max" in spec)
}

# value that can be allowed
ALLOWED_VALUES = {
    column: set(spec["allowed_values"])
    for column, spec in SCHEMA_CONTRACT.items()
    if "allowed_values" in spec
}

# percentage of permissible null values (for specific percentage can be seen at schema contract on OTHER SCHEMA)
MISSINGNESS_THRESHOLDS = {
    column: spec.get("max_null_ratio", 0.0 if not spec["nullable"] else 1.0)
    for column, spec in SCHEMA_CONTRACT.items()
}

print(f"Tipe data: {EXPECTED_DTYPES}\n")
print(f"String coloumn: {STRING_COLUMNS}\n")
print(f"Numerical coloumn: {NUMERIC_BOUNDS}\n")
print(f"Only allowed values: {ALLOWED_VALUES}\n")
print(f"Null value (%) threshold: \n{MISSINGNESS_THRESHOLDS} \n")

Tipe data: {'passengerid': Int64, 'survived': Int64, 'pclass': Int64, 'name': String, 'sex': String, 'sibsp': Int64, 'parch': Int64, 'ticket': String, 'age': Float64, 'fare': Float64, 'cabin': String, 'embarked': String}

String coloumn: ['name', 'sex', 'ticket', 'cabin', 'embarked']

Numerical coloumn: {'sibsp': {'min': 0, 'max': None}, 'parch': {'min': 0, 'max': None}, 'age': {'min': 0, 'max': 100}, 'fare': {'min': 0, 'max': None}}

Only allowed values: {'survived': {0, 1}, 'pclass': {1, 2, 3}, 'sex': {'female', 'male'}, 'embarked': {'S', 'Q', 'C'}}

Null value (%) threshold: 
{'passengerid': 0.0, 'survived': 0.0, 'pclass': 0.0, 'name': 0.0, 'sex': 0.0, 'sibsp': 0.0, 'parch': 0.0, 'ticket': 0.0, 'age': 0.3, 'fare': 0.05, 'cabin': 0.9, 'embarked': 0.05} 



In [11]:
# 
TRAIN_DIFF_COLS = {"survived"}

SCHEMA_TRAIN_BP = COLUMN_ORDER

SCHEMA_TEST_BP = [
    column for column in COLUMN_ORDER if column not in TRAIN_DIFF_COLS
]

print(f"Schema column train: \n{SCHEMA_TRAIN_BP}\n")
print(f"Schema column test: \n{SCHEMA_TEST_BP}")

Schema column train: 
['passengerid', 'survived', 'pclass', 'name', 'sex', 'sibsp', 'parch', 'ticket', 'age', 'fare', 'cabin', 'embarked']

Schema column test: 
['passengerid', 'pclass', 'name', 'sex', 'sibsp', 'parch', 'ticket', 'age', 'fare', 'cabin', 'embarked']


In [12]:
REQUIRED_NON_NULL = {
    "train": [
        column
        for column in SCHEMA_TRAIN_BP
        if not SCHEMA_CONTRACT[column]["nullable"]
    ],
    
    "test": [
        column
        for column in SCHEMA_TEST_BP
        if not SCHEMA_CONTRACT[column]["nullable"]
    ],
}

print(f"Required & Non-null ➡️ [train]: \n{REQUIRED_NON_NULL["train"]} \n")
print(f"Required & Non-null ➡️ [test]: \n{REQUIRED_NON_NULL["test"]} \n")

Required & Non-null ➡️ [train]: 
['passengerid', 'survived', 'pclass', 'name', 'sex', 'sibsp', 'parch', 'ticket'] 

Required & Non-null ➡️ [test]: 
['passengerid', 'pclass', 'name', 'sex', 'sibsp', 'parch', 'ticket'] 



### Align the placement of the expectation header with the raw data.

| Raw Train         | Raw Test      |
| ----------------- | ------------- |
| PassengerId       | PassengerId   |
| Survived          | -             |
| Pclass            | Pclass        |
| Name              | Name          |
| Sex               | Sex           |
| Age               | Age           |
| SibSp             | SibSp         | 
| Parch             | Parch         | 
| Ticket            | Ticket        | 
| Fare              | Fare          | 
| Cabin             | Cabin         | 
| Embarked          | Embarked      |

In [13]:
ADJUSTMENT_TRAIN_COLUMNS = [
    "passengerid",
    "survived",
    "pclass",
    "name",
    "sex",
    "age",
    "sibsp",
    "parch",
    "ticket",
    "fare",
    "cabin",
    "embarked",
]

ADJUSTMENT_TEST_COLUMNS = [
    "passengerid",
    "pclass",
    "name",
    "sex",
    "age",
    "sibsp",
    "parch",
    "ticket",
    "fare",
    "cabin",
    "embarked",
]

TRAIN_COL_BP = [
    column
    for column in ADJUSTMENT_TRAIN_COLUMNS
    if column in SCHEMA_TRAIN_BP
]

TEST_COL_BP = [
    column
    for column in ADJUSTMENT_TEST_COLUMNS
    if column in SCHEMA_TEST_BP
]

print(f"Train column BP: \n{TRAIN_COL_BP}\n")
print(f"Test column BP: \n{TEST_COL_BP}")

Train column BP: 
['passengerid', 'survived', 'pclass', 'name', 'sex', 'age', 'sibsp', 'parch', 'ticket', 'fare', 'cabin', 'embarked']

Test column BP: 
['passengerid', 'pclass', 'name', 'sex', 'age', 'sibsp', 'parch', 'ticket', 'fare', 'cabin', 'embarked']


In [14]:
# Reorder dictionary berdasarkan urutan di list
TRAIN_SCHEMA = {key: SCHEMA_CONTRACT[key] for key in TRAIN_COL_BP if key in SCHEMA_CONTRACT}

print(f"--- ISI TRAIN SCHEMA ---")
print(json.dumps(TRAIN_SCHEMA, indent=2))

--- ISI TRAIN SCHEMA ---
{
  "passengerid": {
    "dtype": "Int64",
    "semantic_type": "primary_key",
    "nullable": false,
    "required": true
  },
  "survived": {
    "dtype": "Int64",
    "semantic_type": "binary_target",
    "allowed_values": [
      0,
      1
    ],
    "nullable": false,
    "required": true
  },
  "pclass": {
    "dtype": "Int64",
    "semantic_type": "categorical",
    "allowed_values": [
      1,
      2,
      3
    ],
    "nullable": false,
    "required": true
  },
  "name": {
    "dtype": "String",
    "semantic_type": "text",
    "nullable": false,
    "required": true
  },
  "sex": {
    "dtype": "String",
    "semantic_type": "categorical",
    "allowed_values": [
      "male",
      "female"
    ],
    "nullable": false,
    "required": true
  },
  "age": {
    "dtype": "Float64",
    "semantic_type": "numeric",
    "min": 0,
    "max": 100,
    "max_null_ratio": 0.3,
    "nullable": true,
    "required": false
  },
  "sibsp": {
    "dtype": "Int6

In [15]:
# Duplikat TRAIN_SCHEMA agar data tidak berubah
TEST_SCHEMA = TRAIN_SCHEMA.copy()

# Hapus kolom target
TEST_SCHEMA.pop("survived", None)

print(f"--- ISI TRAIN SCHEMA ---")
print(json.dumps(TEST_SCHEMA, indent=2))

--- ISI TRAIN SCHEMA ---
{
  "passengerid": {
    "dtype": "Int64",
    "semantic_type": "primary_key",
    "nullable": false,
    "required": true
  },
  "pclass": {
    "dtype": "Int64",
    "semantic_type": "categorical",
    "allowed_values": [
      1,
      2,
      3
    ],
    "nullable": false,
    "required": true
  },
  "name": {
    "dtype": "String",
    "semantic_type": "text",
    "nullable": false,
    "required": true
  },
  "sex": {
    "dtype": "String",
    "semantic_type": "categorical",
    "allowed_values": [
      "male",
      "female"
    ],
    "nullable": false,
    "required": true
  },
  "age": {
    "dtype": "Float64",
    "semantic_type": "numeric",
    "min": 0,
    "max": 100,
    "max_null_ratio": 0.3,
    "nullable": true,
    "required": false
  },
  "sibsp": {
    "dtype": "Int64",
    "semantic_type": "count",
    "min": 0,
    "nullable": false,
    "required": true
  },
  "parch": {
    "dtype": "Int64",
    "semantic_type": "count",
    "min": 

# Validation Rule

## Integrity

In [16]:
from src.ingestion import *

### Source Validation

In [17]:
train_source_check = check_source_file(file_path = TRAIN_RAW, dataset_name = "train")

if train_source_check["status"] != "PASS":
    sys.exit(1)


[INFO] Memulai Pengecekan Dataset: 'train'

[DEBUG] Target:
  ├─ Path      : D:\GIT DATA\titanic_survival_analysis\data\raw\train.csv
  ├─ File      : train.csv
  └─ Dataset   : train

[DEBUG] [1/4] File Existence
  ├─ Expected  : File exists
  └─ Result    : PASS

[DEBUG] [2/4] File Type
  ├─ Expected  : Regular file
  └─ Result    : PASS

[DEBUG] [3/4] File Extension
  ├─ Expected  : .csv
  ├─ Actual    : .csv
  └─ Result    : PASS

[DEBUG] [4/4] File Integrity
  ├─ Size      : 61,194 Bytes (59.76 KB)
  ├─ Readable  : True
  ├─ Modified  : 2026-09-16 15:28:09
  └─ Result    : PASS

----------------------------------------------------------------------
[PASS] VALIDASI SUKSES: File 'train' memenuhi seluruh kriteria.
----------------------------------------------------------------------

[DEBUG] Validation Summary:
  ├─ Dataset     : train
  ├─ File        : train.csv
  ├─ Type        : CSV / Regular File
  ├─ Size        : 59.76 KB
  ├─ Readable    : True
  └─ Result      : 4/4 PASS


In [18]:
test_source_check = check_source_file(file_path = TEST_RAW, dataset_name = "test")

if test_source_check["status"] != "PASS":
    sys.exit(1)


[INFO] Memulai Pengecekan Dataset: 'test'

[DEBUG] Target:
  ├─ Path      : D:\GIT DATA\titanic_survival_analysis\data\raw\test.csv
  ├─ File      : test.csv
  └─ Dataset   : test

[DEBUG] [1/4] File Existence
  ├─ Expected  : File exists
  └─ Result    : PASS

[DEBUG] [2/4] File Type
  ├─ Expected  : Regular file
  └─ Result    : PASS

[DEBUG] [3/4] File Extension
  ├─ Expected  : .csv
  ├─ Actual    : .csv
  └─ Result    : PASS

[DEBUG] [4/4] File Integrity
  ├─ Size      : 28,629 Bytes (27.96 KB)
  ├─ Readable  : True
  ├─ Modified  : 2026-09-16 15:28:09
  └─ Result    : PASS

----------------------------------------------------------------------
[PASS] VALIDASI SUKSES: File 'test' memenuhi seluruh kriteria.
----------------------------------------------------------------------

[DEBUG] Validation Summary:
  ├─ Dataset     : test
  ├─ File        : test.csv
  ├─ Type        : CSV / Regular File
  ├─ Size        : 27.96 KB
  ├─ Readable    : True
  └─ Result      : 4/4 PASS


### Source Fingerprint

In [19]:
train_fingerprint_check = check_source_fingerprint(file_path = TRAIN_RAW, dataset_name = "train")


[INFO] SOURCE FINGERPRINT: 'train'

[DEBUG] Target:
  ├─ Path        : D:\GIT DATA\titanic_survival_analysis\data\raw\train.csv
  ├─ File        : train.csv
  ├─ Dataset     : train
  └─ Fingerprint : D:\GIT DATA\titanic_survival_analysis\metadata\source_fingerprint.json

[DEBUG] [1/5] Source File
  ├─ Expected  : Existing regular file
  ├─ Actual    : File ditemukan
  └─ Result    : PASS

[DEBUG] [2/5] SHA-256 Calculation
  ├─ Algorithm : SHA-256
  ├─ Source    : train.csv
  ├─ SHA-256   : 7d118fef8b6ccf7f81111877bc388536f7b1e498a655e3d649d19aaa010e9f6f
  └─ Result    : PASS

[DEBUG] [3/5] Previous Fingerprint
  ├─ Metadata  : D:\GIT DATA\titanic_survival_analysis\metadata\source_fingerprint.json
  ├─ Previous  : Found
  └─ Result    : PASS

[DEBUG] [4/5] Source Identity
  ├─ Dataset:
  │  ├─ Previous : train
  │  ├─ Current  : train
  │  └─ Match    : True
  ├─ File Name:
  │  ├─ Previous : train.csv
  │  ├─ Current  : train.csv
  │  └─ Match    : True
  ├─ File Path:
  │  ├─ Previo

In [20]:
test_fingerprint_check = check_source_fingerprint(file_path = TEST_RAW, dataset_name = "test")


[INFO] SOURCE FINGERPRINT: 'test'

[DEBUG] Target:
  ├─ Path        : D:\GIT DATA\titanic_survival_analysis\data\raw\test.csv
  ├─ File        : test.csv
  ├─ Dataset     : test
  └─ Fingerprint : D:\GIT DATA\titanic_survival_analysis\metadata\source_fingerprint.json

[DEBUG] [1/5] Source File
  ├─ Expected  : Existing regular file
  ├─ Actual    : File ditemukan
  └─ Result    : PASS

[DEBUG] [2/5] SHA-256 Calculation
  ├─ Algorithm : SHA-256
  ├─ Source    : test.csv
  ├─ SHA-256   : 56023b9948236f3c7a1c9448fcf418b283e109ef177fa8c7e069158dd7dd52b2
  └─ Result    : PASS

[DEBUG] [3/5] Previous Fingerprint
  ├─ Metadata  : D:\GIT DATA\titanic_survival_analysis\metadata\source_fingerprint.json
  ├─ Previous  : Found
  └─ Result    : PASS

[DEBUG] [4/5] Source Identity
  ├─ Dataset:
  │  ├─ Previous : test
  │  ├─ Current  : test
  │  └─ Match    : True
  ├─ File Name:
  │  ├─ Previous : test.csv
  │  ├─ Current  : test.csv
  │  └─ Match    : True
  ├─ File Path:
  │  ├─ Previous : D:\G

### Format & Structure Validation

In [21]:
train_structural_check = check_csv_structure(file_path = TRAIN_RAW, dataset_name = "train", expected_columns = TRAIN_COL_BP)

if train_structural_check["status"] != "PASS":
    sys.exit(1)


[INFO] Memulai Structural Validation Dataset: 'train'

[DEBUG] Target:
  ├─ Path      : D:\GIT DATA\titanic_survival_analysis\data\raw\train.csv
  ├─ File      : train.csv
  └─ Dataset   : train

[DEBUG] [1/4] CSV Readability
  ├─ Expected  : File CSV dapat dibaca
  ├─ Actual    : File memiliki isi
  └─ Result    : PASS

[DEBUG] [2/4] CSV Header
  ├─ Expected  : Header tersedia dan seluruh kolom memiliki nama
  ├─ Actual    : 12 kolom
  └─ Result    : PASS

[DEBUG] [3/4] Column Structure
  ├─ Expected  : Struktur kolom sesuai expected schema
  ├─ Actual    : 12 kolom
  ├─ Matching  : Name + order + uniqueness
  └─ Result    : PASS

[DEBUG] [4/4] Row Structure
  ├─ Expected  : 12 field per row
  ├─ Actual    : 891 row diperiksa
  ├─ Invalid   : 0 row
  └─ Result    : PASS

----------------------------------------------------------------------
[PASS] VALIDASI SUKSES: Struktur CSV 'train' valid.
----------------------------------------------------------------------

[DEBUG] Validation Su

In [22]:
test_structural_check = check_csv_structure(file_path = TEST_RAW, dataset_name = "test", expected_columns = TEST_COL_BP)

if test_structural_check["status"] != "PASS":
    sys.exit(1)


[INFO] Memulai Structural Validation Dataset: 'test'

[DEBUG] Target:
  ├─ Path      : D:\GIT DATA\titanic_survival_analysis\data\raw\test.csv
  ├─ File      : test.csv
  └─ Dataset   : test

[DEBUG] [1/4] CSV Readability
  ├─ Expected  : File CSV dapat dibaca
  ├─ Actual    : File memiliki isi
  └─ Result    : PASS

[DEBUG] [2/4] CSV Header
  ├─ Expected  : Header tersedia dan seluruh kolom memiliki nama
  ├─ Actual    : 11 kolom
  └─ Result    : PASS

[DEBUG] [3/4] Column Structure
  ├─ Expected  : Struktur kolom sesuai expected schema
  ├─ Actual    : 11 kolom
  ├─ Matching  : Name + order + uniqueness
  └─ Result    : PASS

[DEBUG] [4/4] Row Structure
  ├─ Expected  : 11 field per row
  ├─ Actual    : 418 row diperiksa
  ├─ Invalid   : 0 row
  └─ Result    : PASS

----------------------------------------------------------------------
[PASS] VALIDASI SUKSES: Struktur CSV 'test' valid.
----------------------------------------------------------------------

[DEBUG] Validation Summary

### Schema Validation

In [23]:
train_schema_validation = check_csv_schema(file_path = TRAIN_RAW, dataset_name = "train", schema = TRAIN_SCHEMA)

if train_schema_validation["status"] != "PASS":
    sys.exit(1)


[INFO] Memulai Schema Validation Dataset: 'train'

[DEBUG] Target:
  ├─ Path      : D:\GIT DATA\titanic_survival_analysis\data\raw\train.csv
  ├─ File      : train.csv
  └─ Dataset   : train

[DEBUG] Loading Dataset
  ├─ Columns   : 12
  ├─ Rows      : 891
  └─ Result    : PASS

[DEBUG] Column Mapping
  ├─ Mode      : CASE-INSENSITIVE
  ├─ CSV Header:
  │  ├─ PassengerId
  │  ├─ Survived
  │  ├─ Pclass
  │  ├─ Name
  │  ├─ Sex
  │  ├─ Age
  │  ├─ SibSp
  │  ├─ Parch
  │  ├─ Ticket
  │  ├─ Fare
  │  ├─ Cabin
  │  ├─ Embarked
  └─ Schema Mapping:
     ├─ passengerid → PassengerId
     ├─ survived → Survived
     ├─ pclass → Pclass
     ├─ name → Name
     ├─ sex → Sex
     ├─ age → Age
     ├─ sibsp → SibSp
     ├─ parch → Parch
     ├─ ticket → Ticket
     ├─ fare → Fare
     ├─ cabin → Cabin
     ├─ embarked → Embarked

[DEBUG] [1/8] Data Type Validation
  ├─ Expected  : Tipe data sesuai expected schema
  ├─ Checked   : 12 kolom
  ├─ Invalid   : 0
  └─ Result    : PASS

[DEBUG] [2/8] 

In [24]:
test_schema_validation = check_csv_schema(file_path = TEST_RAW, dataset_name = "test", schema = TEST_SCHEMA)

if test_schema_validation["status"] != "PASS":
    sys.exit(1)


[INFO] Memulai Schema Validation Dataset: 'test'

[DEBUG] Target:
  ├─ Path      : D:\GIT DATA\titanic_survival_analysis\data\raw\test.csv
  ├─ File      : test.csv
  └─ Dataset   : test

[DEBUG] Loading Dataset
  ├─ Columns   : 11
  ├─ Rows      : 418
  └─ Result    : PASS

[DEBUG] Column Mapping
  ├─ Mode      : CASE-INSENSITIVE
  ├─ CSV Header:
  │  ├─ PassengerId
  │  ├─ Pclass
  │  ├─ Name
  │  ├─ Sex
  │  ├─ Age
  │  ├─ SibSp
  │  ├─ Parch
  │  ├─ Ticket
  │  ├─ Fare
  │  ├─ Cabin
  │  ├─ Embarked
  └─ Schema Mapping:
     ├─ passengerid → PassengerId
     ├─ pclass → Pclass
     ├─ name → Name
     ├─ sex → Sex
     ├─ age → Age
     ├─ sibsp → SibSp
     ├─ parch → Parch
     ├─ ticket → Ticket
     ├─ fare → Fare
     ├─ cabin → Cabin
     ├─ embarked → Embarked

[DEBUG] [1/8] Data Type Validation
  ├─ Expected  : Tipe data sesuai expected schema
  ├─ Checked   : 11 kolom
  ├─ Invalid   : 0
  └─ Result    : PASS

[DEBUG] [2/8] Nullability Validation
  ├─ Expected  : NULL hany

### Data Quality Validation

### Artifact Validation

### Provenance & Audit Validation